## Tools

In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash'
)
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.3'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x000002649B540AF0>, default_metadata=(), model_kwargs={})

In [4]:
response = model.invoke("What is a parrot?")
response.text

'A **parrot** is a type of bird belonging to the order **Psittaciformes**. They are found mostly in tropical and subtropical regions around the world, particularly in South America, Australia, Africa, and parts of Asia.\n\nHere are their key characteristics:\n\n1.  **Distinctive Beak:** Parrots have a strong, curved, hooked beak. The upper mandible is larger and curves over the lower one, making it excellent for cracking nuts, seeds, and climbing.\n2.  **Zygodactyl Feet:** Their feet have two toes pointing forward and two pointing backward. This arrangement gives them a powerful grip for climbing, perching, and manipulating objects (like holding food).\n3.  **Often Vibrantly Colored:** Many parrot species are known for their spectacular, often bright and varied plumage, though some are more subdued in color.\n4.  **Intelligence:** Parrots are considered one of the most intelligent bird species. They have complex problem-solving abilities and can learn and remember.\n5.  **Mimicry and V

In [5]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather in a given location."""
    # For demonstration purposes, we'll return a dummy weather report.
    return f"The current weather in {location} is sunny with a temperature of 75°F."
model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("What is the weather in Boston?")
print (response)
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']} with arguments: {tool_call['args']}")

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'2f6900da-6579-412b-969e-158968c47e6a': 'CvQBARFNMg8ndEzxOAt3j86JToMOOGBpIlLTFTDjKo5laZ3WtPnT14PPVuCGZZXIutNMtRwkD9PcaPbF05AyAuEW5PXv8mUzmJrlgTma6+mt+87Lvo9OMCMpL05rAWAHYfDagwbpM0KU4sSk/+Sp4O4yI9Go6/c9r4uWXSZ+yTYmRV8/21IdLefURImxFO1OomrizhmdxlGIVdMoZ4F79zBTVw5V6dgFHx/hGjz6N0hXI9rOjdAU5Yjwpv0FIorNSsIaUkD+jYPDt+kwUOxiBmPFoi7bdhrfwc7G/puJ7WZ790kxjaQpPKPGwYo6OKP03IEHjxD/1A=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0092b-3db4-7272-95ce-ac21834c866b-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '2f6900da-6579-412b-969e-158968c47e6a', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 49, 'output_tokens': 64, 'total_tokens': 113, 'input_token_details': {'cache_read': 0}, 

### Tools Execution loop

In [8]:
# Step 1 - Model generates tool calls

messages = [{ "role": "user", "content": "What is the weather in Boston?" }]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

In [10]:
# Step 2 - Execute the tool calls and get the results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

In [11]:
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The current weather in Boston is sunny with a temperature of 75°F.


In [12]:
messages

[{'role': 'user', 'content': 'What is the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'d7658395-4529-4609-a977-49d1d3b63d34': 'CukBARFNMg/v5n7wJXfeCKqGyULvAJvdgBRQ3jjg/y0zBU5MtD1QXnnceGj3XnZbk9FdKEl1x+GSqO2zbjh3g/CXvJefz9vTiRQQKj1vI+Y8Vqh18x0VnRN2MV877EScKYRSb06inZJcRRj9zz+HJRkJ77NYyCK2uuZjYjNByfiXxkDICxY01W1/rkonCxLOp2DdkFJk0xw+uRiAsXgcGkcfYTCu19FnMgOTNlVMdShGcjrySRr6ds5dZ/jO4bdM2mc3IyDv9WPV/Xtb3j7l7Gk1MIL7LIMgdMxQhsO9NxeiBPxefwqx3NI1pqA='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0092b-89d9-7750-ae03-ddeaab1abfe9-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'd7658395-4529-4609-a977-49d1d3b63d34', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 61